In [60]:
from dotenv import load_dotenv
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableParallel , RunnablePassthrough , RunnableLambda

In [43]:
load_dotenv()

embedding_model = GoogleGenerativeAIEmbeddings(model= 'gemini-embedding-2-preview')

model = ChatGoogleGenerativeAI(model = 'gemini-3.1-flash-lite-preview')

In [ ]:
loader = WikipediaLoader(
    query='What is a Machine Learning' , 
    lang = 'en',
    load_max_docs=19
)

In [6]:
document = loader.load()

In [10]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1200 , 
    chunk_overlap = 0
)

In [11]:
chunk = splitter.split_documents(document)

In [12]:
len(chunk)

92

In [16]:
chroma_store = Chroma.from_documents(
    embedding = embedding_model , 
    documents = chunk
)

In [23]:
retriever = chroma_store.as_retriever(
    search_type = 'mmr' , 
    search_kwargs = {'k': 6}
)

In [28]:
query = 'What is ANN'

In [30]:
retrive_document =retriever.invoke(query)

In [ ]:
context_text = '\n\n'.join(d.page_content for d in retrive_document)

In [40]:
len(context_text)

4562

In [53]:
parser = StrOutputParser()

In [ ]:
def format(retrive_document):
    context_text = '\n\n'.join(doc.page_content for doc in retrive_document)
    return context_text

In [71]:
parallel_chain = RunnableParallel({
    'context' : retriever | RunnableLambda(format)   , 
    'query' : RunnablePassthrough()
})

In [72]:
prompt = PromptTemplate( 
    template='Hey , you are my assistant for now , i am sharing a context and a question and you have to answer and if text is insufficent then just say , I don\'t know \n context --- > {context} , \n question -- > {query} ' ,
    
    input_variables=['query' , 'text'] )

In [73]:
model_chain = prompt | model |parser

In [75]:
chain = parallel_chain | model_chain 

In [77]:
result = chain.invoke('Whar is ANN')

In [78]:
result

'An ANN (Artificial Neural Network) is a computational model inspired by the structure and functions of biological neural networks, used in machine learning. It consists of connected units or nodes called artificial neurons, which process signals and are connected by edges that model synapses in the brain.'

In [76]:
chain.get_graph().print_ascii()

             +------------------------------+          
             | Parallel<context,query>Input |          
             +------------------------------+          
                    **               ***               
                 ***                    **             
               **                         ***          
+----------------------+                     **        
| VectorStoreRetriever |                      *        
+----------------------+                      *        
            *                                 *        
            *                                 *        
            *                                 *        
       +--------+                     +-------------+  
       | format |                     | Passthrough |  
       +--------+***                  +-------------+  
                    **               **                
                      ***         ***                  
                         **     **              